# 🦥 Unsloth Practical Lab: Ultra-Fast & Memory-Efficient LLM Fine-Tuning
> **A complete, end-to-end master notebook** demonstrating how to fine-tune Large Language Models (LLMs) **2× to 5× faster** with **50% to 80% less VRAM** using **Unsloth**, **Hugging Face Transformers**, and **TRL**.

---

### 📋 What You Will Learn
| Section | Topic | Core Concept |
|---|---|---|
| **1. Environment Setup** | Installation & Diagnostics | Install Unsloth, verify Triton/CUDA runtime |
| **2. Base Model Loading** | `FastLanguageModel.from_pretrained` | 4-bit quantization, memory mapping, RoPE scaling |
| **3. LoRA Injection** | `FastLanguageModel.get_peft_model` | Target modules, rank, alpha, zero-dropout optimization |
| **4. Data Engineering** | Alpaca Formatting | Prompt templates, tokenization, EOS token handling |
| **5. SFT Training Loop** | TRL `SFTTrainer` & `SFTConfig` | 8-bit AdamW, gradient accumulation, VRAM benchmarking |
| **6. Fast Inference** | `FastLanguageModel.for_inference` | 2× inference speedup, latency & tokens/sec profiling |
| **7. Multi-Format Export** | LoRA, Merged 16-bit, GGUF | Exporting to vLLM, Ollama, and llama.cpp |
| **8. Benchmarks & Long Context** | Unsloth vs Stock PyTorch | VRAM comparison, 20K–300K+ context window scaling |
| **9. Advanced Workflows** | Reasoning RL (GRPO) | DeepSeek R1-style alignment on low VRAM |

---

### 🧠 Architectural Blueprint

```
┌────────────────────────────────────────────────────────────────────────┐
│                        DEEP LEARNING PLATFORMS                         │
│                  PyTorch  /  CUDA  /  Triton  /  ROCm                  │
└───────────────────────────────────┬────────────────────────────────────┘
                                    │
┌───────────────────────────────────▼────────────────────────────────────┐
│                       HIGH-LEVEL ECOSYSTEM                             │
│       Hugging Face Transformers  │  PEFT  │  TRL (SFT / DPO / PPO)     │
└───────────────────────────────────┬────────────────────────────────────┘
                                    │
┌───────────────────────────────────▼────────────────────────────────────┐
│                   UNSLOTH OPTIMIZATION ENGINE                          │
│  FastLanguageModel  │  Custom Triton Kernels  │  Manual Backprop       │
│  Fused RMSNorm/RoPE │  Cross-Entropy Engine   │  Neat Sequence Packing │
└───────────────────────────────────┬────────────────────────────────────┘
                                    │
┌───────────────────────────────────▼────────────────────────────────────┐
│                        HARDWARE ACCELERATION                           │
│     NVIDIA GPUs (T4, V100, RTX 3090/4090, A100, H100, B200)            │
└────────────────────────────────────────────────────────────────────────┘
```


---
## 1. ⚙️ Environment Setup & Hardware Diagnostics

Unsloth relies on low-level **CUDA** and **OpenAI Triton** kernels to accelerate matrix multiplications and bypass PyTorch autograd graph overhead.

### Installation Instructions
- In Google Colab or Kaggle: Use the pre-built unsloth wheel matching your CUDA version.
- Unsloth integrates directly with `transformers`, `peft`, and `trl`.


In [ ]:
# =============================================================================
# 1. INSTALLATION (Colab / Kaggle / Local)
# =============================================================================
# Note: In standard Google Colab, uncomment and run:
# !pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
# !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# !pip install --no-deps trl peft accelerate bitsandbytes

import os
import sys

print("Python version:", sys.version)


In [ ]:
# =============================================================================
# 2. HARDWARE & RUNTIME CONFIGURATION
# =============================================================================
import torch
import random
import numpy as np

# Ensure reproducible training
SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Verify GPU runtime
assert torch.cuda.is_available(), "❌ CUDA GPU not detected! Please enable GPU runtime."

gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
is_bf16_supported = torch.cuda.is_bf16_supported()

print("=" * 60)
print(f"🚀 GPU Name:              {gpu_name}")
print(f"💾 Total VRAM:            {gpu_memory_gb:.2f} GB")
print(f"⚡ Bfloat16 Supported:    {is_bf16_supported}")
print(f"🎯 Random Seed:           {SEED}")
print("=" * 60)


---
## 2. 📦 Loading Base Model with `FastLanguageModel.from_pretrained`

Unsloth replaces `AutoModelForCausalLM` with `FastLanguageModel`:
- **Pre-quantized 4-bit weights** directly mapped to memory (4× faster download & load).
- **Custom RoPE Scaling** configured for arbitrary context lengths.
- Native precision management (BF16 on Ampere/Hopper/Ada, FP16 on Turing).


In [ ]:
# =============================================================================
# 3. LOAD BASE MODEL & TOKENIZER
# =============================================================================
try:
    from unsloth import FastLanguageModel
    UNSLOTH_AVAILABLE = True
except ImportError:
    print("⚠️ unsloth library not installed in this environment. Simulating fallback.")
    UNSLOTH_AVAILABLE = False

# Maximum sequence length to support (Unsloth supports up to 128K context)
max_seq_length = 2048

# Data type: None = auto-detect (BF16 on Ampere/Hopper, FP16 on older GPUs)
dtype = None

# 4-bit quantization flag (reduces memory consumption by ~75%)
load_in_4bit = True

# Pre-quantized models available from unsloth:
# - "unsloth/tinyllama-bnb-4bit"
# - "unsloth/Meta-Llama-3.1-8B-bnb-4bit"
# - "unsloth/Qwen2.5-7B-bnb-4bit"
# - "unsloth/gemma-2-2b-bnb-4bit"
BASE_MODEL_NAME = "unsloth/tinyllama-bnb-4bit"

print(f"Loading {BASE_MODEL_NAME} (max_seq_length={max_seq_length})...")

if UNSLOTH_AVAILABLE:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL_NAME,
        max_seq_length=max_seq_length,
        dtype=dtype,
        load_in_4bit=load_in_4bit,
    )
    print("✅ Model and tokenizer loaded successfully with Unsloth!")
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    q_cfg = BitsAndBytesConfig(load_in_4bit=True)
    tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T")
    model = AutoModelForCausalLM.from_pretrained(
        "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
        quantization_config=q_cfg,
        device_map="auto"
    )
    print("✅ Fallback model loaded using standard Transformers + BitsAndBytes.")


In [ ]:
# =============================================================================
# 4. INSPECT MODEL ARCHITECTURE & DEVICE
# =============================================================================
param_sample = next(model.parameters())
print(f"Model Type:      {type(model).__name__}")
print(f"Device:          {param_sample.device}")
print(f"Dtype:           {param_sample.dtype}")
print(f"Vocab Size:      {len(tokenizer)}")
print(f"PAD Token:       {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")
print(f"EOS Token:       {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")


---
## 3. 🎯 Applying LoRA Adapters with `FastLanguageModel.get_peft_model`

Unsloth attaches Low-Rank Adaptation (LoRA) matrices into the model:
- **Target Modules:** We target **all 7 linear projection layers** (`q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`) for maximum learning capacity.
- **Zero Dropout (`lora_dropout=0.0`):** Enables custom fused Triton kernels.
- **Smart Gradient Checkpointing (`use_gradient_checkpointing="unsloth"`):** Slashes memory overhead with near-zero recomputation latency.


In [ ]:
# =============================================================================
# 5. INJECT OPTIMIZED LORA ADAPTERS (PEFT)
# =============================================================================
if UNSLOTH_AVAILABLE:
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,                                    # LoRA rank (8, 16, 32, 64)
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",    # Self-attention projections
            "gate_proj", "up_proj", "down_proj"       # MLP / FFN projections
        ],
        lora_alpha=32,                           # Scaling factor (alpha = 2 * r)
        lora_dropout=0.0,                        # 0.0 enables custom fused Triton kernels!
        bias="none",                             # "none" = do not train bias parameters
        use_gradient_checkpointing="unsloth",    # Unsloth custom smart checkpointing
        random_state=SEED,
        use_rslora=False,                        # Rank-stabilized LoRA flag
        loftq_config=None,
    )
    print("✅ LoRA adapters injected via FastLanguageModel!")
else:
    from peft import LoraConfig, get_peft_model, TaskType
    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none"
    )
    model = get_peft_model(model, lora_cfg)
    print("✅ LoRA adapters injected via PEFT!")


In [ ]:
# =============================================================================
# 6. TRAINABLE PARAMETER SUMMARY
# =============================================================================
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params     = sum(p.numel() for p in model.parameters())
trainable_pct    = 100 * trainable_params / total_params

print("=" * 60)
print(f"Total Parameters:       {total_params:,}")
print(f"Trainable Parameters:   {trainable_params:,}")
print(f"Frozen Parameters:      {total_params - trainable_params:,}")
print(f"Trainable Percentage:   {trainable_pct:.2f}%")
print("=" * 60)


---
## 4. 📝 Data Engineering & Alpaca Formatting

We convert instruction data into standard Alpaca format:
```
Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}<eos>
```

> **CRITICAL RULE:** The `<eos>` (End Of Sequence) token **MUST** be appended to every training response. Without it, the model will never learn when to terminate generation, leading to infinite rambling!


In [ ]:
# =============================================================================
# 7. DATASET FORMATTING (Alpaca Template)
# =============================================================================
from datasets import Dataset, load_dataset

EOS_TOKEN = tokenizer.eos_token

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        # Crucial: Must append EOS_TOKEN at the end!
        text = alpaca_prompt.format(instruction, input_text, output) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

# Create a practical domain dataset (Linux terminal & Systems Engineering)
sample_data = {
    "instruction": [
        "Explain what the grep command does in Linux.",
        "How do I find all files ending in .log modified in the last 24 hours?",
        "What does chmod 755 do to a file?",
        "Explain the difference between a hard link and a symbolic link.",
        "How do you monitor live GPU utilization on Linux?"
    ],
    "input": [
        "",
        "Context: System administration",
        "",
        "",
        "Hardware: NVIDIA GPUs"
    ],
    "output": [
        "The grep command searches for patterns within files or standard input using regular expressions and prints matching lines. Common flags include -i (case-insensitive), -r (recursive), and -n (show line numbers).",
        "Use the find command: `find /path/to/search -name '*.log' -mtime -1`. The -mtime -1 flag targets files modified within the past 24 hours.",
        "chmod 755 grants Read, Write, and Execute (rwx = 7) permissions to the file owner, and Read and Execute (r-x = 5) permissions to the group and all other users.",
        "A hard link points directly to the inode of the file, sharing data and surviving the deletion of the original link. A symbolic (soft) link points to the file path; if the target file is deleted, the symlink breaks.",
        "Use `nvidia-smi` for instantaneous stats, or `watch -n 1 nvidia-smi` to refresh every second. For detailed metrics, tools like nvtop or gpustat can be used."
    ]
}

dataset = Dataset.from_dict(sample_data)
dataset = dataset.map(formatting_prompts_func, batched=True)

print("✅ Formatted Dataset Sample:")
print("-" * 60)
print(dataset["text"][0])
print("-" * 60)


---
## 5. 🚀 Supervised Fine-Tuning (SFT) with TRL `SFTTrainer`

We configure **TRL's `SFTTrainer`** combined with Unsloth optimizations:
- **`optim="adamw_8bit"`:** 8-bit AdamW optimizer cuts optimizer states memory by **75%** (from 8 bytes to 2 bytes per parameter).
- **`gradient_accumulation_steps=4`:** Simulates larger batch sizes without VRAM spikes.
- **Native Precision:** `bf16=True` on Ampere/Hopper/Ada GPUs.
- **Memory & Latency Profiling:** Tracking CPU RAM and peak GPU VRAM before and after training.


In [ ]:
# =============================================================================
# 8. BENCHMARK TRACKING & SFTTRAINER SETUP
# =============================================================================
import time
import psutil

# Track system state before training
process = psutil.Process()
cpu_ram_before_gb = process.memory_info().rss / (1024**3)
torch.cuda.reset_peak_memory_stats()
torch.cuda.empty_cache()
vram_before_gb = torch.cuda.memory_reserved() / (1024**3)

print(f"Initial CPU RAM: {cpu_ram_before_gb:.2f} GB")
print(f"Initial GPU VRAM: {vram_before_gb:.2f} GB")

from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./unsloth_outputs",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,      # Effective batch size = 2 * 4 = 8
    warmup_steps=5,
    max_steps=30,                       # Fast demonstration run
    learning_rate=2e-4,                 # 2e-4 standard for LoRA
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=5,
    optim="adamw_8bit",                 # 8-bit AdamW optimizer saves ~2GB VRAM
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=SEED,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=1,
    packing=False,                      # Set True to enable Unsloth Neat Sequence Packing
    args=training_args,
)

print("✅ SFTTrainer initialized successfully!")


In [ ]:
# =============================================================================
# 9. EXECUTE TRAINING & COMPUTE HARDWARE METRICS
# =============================================================================
train_start_time = time.time()

print("=" * 60)
print("STARTING TRAINING RUN")
print("=" * 60)

trainer_stats = trainer.train()

train_end_time = time.time()
training_time_sec = train_end_time - train_start_time

# Compute hardware usage metrics
cpu_ram_after_gb = process.memory_info().rss / (1024**3)
peak_gpu_vram_gb = torch.cuda.max_memory_reserved() / (1024**3)

print("=" * 60)
print("📊 TRAINING PERFORMANCE STATS")
print("=" * 60)
print(f"⏱️ Total Training Time:     {training_time_sec:.2f} seconds")
print(f"💾 Peak GPU VRAM Reserved:   {peak_gpu_vram_gb:.2f} GB")
print(f"🧠 CPU RAM Delta:            {cpu_ram_after_gb - cpu_ram_before_gb:.2f} GB")
print(f"📉 Final Training Loss:      {trainer_stats.training_loss:.4f}")
print("=" * 60)


---
## 6. ⚡ Fast Inference with `FastLanguageModel.for_inference`

Unsloth includes an integrated **2× inference speedup engine**:
- Calling `FastLanguageModel.for_inference(model)` optimizes the forward pass and KV-cache.
- Enables high-throughput sampling without requiring an external server like vLLM for local testing.


In [ ]:
# =============================================================================
# 10. INFERENCE ACCELERATION & EVALUATION
# =============================================================================
if UNSLOTH_AVAILABLE:
    FastLanguageModel.for_inference(model)
else:
    model.eval()

test_prompt = alpaca_prompt.format(
    "How do you check available disk space in Linux?",
    "",
    ""
)

inputs = tokenizer([test_prompt], return_tensors="pt").to(model.device)

torch.cuda.synchronize()
start_gen = time.time()

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        use_cache=True,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

torch.cuda.synchronize()
end_gen = time.time()

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
num_generated_tokens = outputs.shape[-1] - inputs["input_ids"].shape[-1]
tokens_per_sec = num_generated_tokens / (end_gen - start_gen)

print("=" * 60)
print("🤖 MODEL GENERATION OUTPUT:")
print("=" * 60)
print(generated_text)
print("-" * 60)
print(f"⚡ Generation Speed: {tokens_per_sec:.2f} tokens/second")
print("=" * 60)


---
## 7. 💾 Multi-Format Export Freedom

Once fine-tuning is complete, Unsloth allows you to export your model into **any desired format**:

```
                     ┌──────────────────────────────────────────────┐
                     │          TRAINED UNSLOTH MODEL               │
                     └──────────────────────┬───────────────────────┘
                                            │
         ┌───────────────────┬──────────────┴───────┬───────────────────┐
         ▼                   ▼                      ▼                   ▼
┌─────────────────┐ ┌─────────────────┐ ┌─────────────────┐ ┌─────────────────┐
│  LoRA Adapters  │ │ Merged 16-bit   │ │ Merged 4-bit    │ │  GGUF Format    │
│  save_pretrained│ │ (vLLM / SGLang) │ │ (Hugging Face)  │ │ (Ollama/llama.cpp│
│  Size: ~50MB    │ │ Size: ~16GB     │ │ Size: ~5.5GB    │ │ Size: ~4.5GB    │
└─────────────────┘ └─────────────────┘ └─────────────────┘ └─────────────────┘
```


In [ ]:
# =============================================================================
# 11. SAVE ADAPTERS & EXPORT MODELS
# =============================================================================
import os

LORA_PATH = "./unsloth_lora_adapter"
os.makedirs(LORA_PATH, exist_ok=True)

# 1. Save LoRA Adapters (~50MB to ~150MB)
model.save_pretrained(LORA_PATH)
tokenizer.save_pretrained(LORA_PATH)
print(f"✅ LoRA adapter saved to: {LORA_PATH}")

# 2. Exporting to Merged 16-bit or GGUF (Unsloth native commands):
# if UNSLOTH_AVAILABLE:
#     # Save merged 16-bit model for vLLM / Hugging Face Hub:
#     model.save_pretrained_merged("./merged_16bit_model", tokenizer, save_method="merged_16bit")
#
#     # Save directly to GGUF format for Ollama / llama.cpp:
#     model.save_pretrained_gguf("./model_gguf", tokenizer, quantization_method="q4_k_m")

print("To deploy in Ollama:")
print("1. Create Modelfile: FROM ./model_gguf/unsloth.Q4_K_M.gguf")
print("2. Run terminal command: ollama create my-custom-model -f Modelfile")
print("3. Chat with model: ollama run my-custom-model")


---
## 8. 📊 Comprehensive Benchmark: Unsloth vs Standard Hugging Face

### Context Scaling on LLaMA-3.1 (8B)

| Hardware | Standard PyTorch + HF | Unsloth Accelerated Engine |
|---|---|---|
| **8 GB GPU** (RTX 3070) | ❌ **OOM Crash** | **~3,000 tokens** |
| **12 GB GPU** (RTX 3060) | ❌ **OOM Crash** | **~21,000 tokens** |
| **16 GB GPU** (Colab T4) | ~2,048 tokens max | **~40,000 tokens** |
| **24 GB GPU** (RTX 4090) | ~8,192 tokens max | **~78,000 tokens** |
| **80 GB GPU** (A100 / H100) | ~28,000 tokens max | **Up to ~340,000 tokens!** |

### Efficiency Comparison

| Dimension | Standard Hugging Face | Unsloth Engine |
|---|---|---|
| **Training Speed** | Baseline (1.0×) | **2.0× to 5.0× faster** |
| **VRAM Consumption** | High (full autograd graph) | **50% to 80% lower** |
| **Backpropagation** | PyTorch Autograd | **Manual Triton Backprop** |
| **Accuracy Degradation** | 0% (Reference) | **0% (Exact Math Guarantee)** |


---
## 9. 🧠 Advanced Workflows: Reasoning RL (GRPO) & Summary

Unsloth provides native support for **Group Relative Policy Optimization (GRPO)** — the reinforcement learning technique behind **DeepSeek-R1**:
- Generates multiple candidate completions for each prompt.
- Evaluates outputs against reward functions (e.g. format compliance, mathematical correctness).
- Calculates group-relative advantages without needing a separate Critic/Value network.
- **Unsloth cuts GRPO VRAM by 80%**, making reasoning model fine-tuning possible on a single 16GB or 24GB GPU!

---

### ✅ Module Completion Checklist
- [x] Environment diagnostics & CUDA GPU verification
- [x] 4-bit base model loading with `FastLanguageModel.from_pretrained`
- [x] Parameter-efficient LoRA injection targeting all 7 linear projections
- [x] Alpaca prompt formatting with mandatory `<eos>` termination
- [x] SFT training loop with 8-bit AdamW optimizer and gradient accumulation
- [x] 2× accelerated inference with `FastLanguageModel.for_inference`
- [x] Multi-format model saving (LoRA, Merged 16-bit, and GGUF)
